In [37]:
import sys
from pathlib import Path
sys.path.insert(0,str(Path().resolve().parent))
import pandas as pd

In [ ]:
#connection to the DB (main.py)
from src.utils.db import connect_to_db

#instanciating an engine
engine= connect_to_db()

In [ ]:
#loading table
#query(main.py)
query_f1_4 = """
    SELECT * FROM bronze.f1_4_air_releases_facilities
    WHERE "countryName"  = 'Belgium' AND "Pollutant"='Carbon dioxide (CO2)' AND "reportingYear" BETWEEN 2016 AND 2024
"""
#use of general function in main
f1_4_be_CO2= pd.read_sql(query_f1_4, con=engine)


print(f"F1_4 Belgique : {f1_4_be_CO2.shape}")


In [ ]:
f1_4_be_CO2.columns

In [ ]:
#nombre d'années manquantes
f1_4_be_CO2.isnull().sum()

In [40]:
#deleting unnecessary columns
col_to_keep=['reportingYear', 'EPRTR_SectorCode',
       'EPRTR_SectorName', 'FacilityInspireId',
       'facilityName', 'city', 'Longitude', 'Latitude',
       'Releases']
f1_4_be_CO2_mod=f1_4_be_CO2[col_to_keep].copy()

In [30]:
f1_4_be_CO2=f1_4_be_CO2.rename(str.lower, axis='columns').rename(columns={'releases':"emitted_co2_kg"})

In [ ]:
f1_4_be_CO2.head()

In [32]:
f1_4_be_CO2.isnull().sum()

publicationdate                   0
countryname                       0
reportingyear                     0
eprtr_sectorcode                  1
eprtr_sectorname                  1
eprtranneximainactivity           1
facilityinspireid                 0
facilityname                      0
city                              7
longitude                         0
latitude                          0
addressconfidentialityreason    597
targetrelease                     0
pollutant                         0
emitted_co2_kg                    0
confidentialityreason           597
dtype: int64

In [33]:
f1_4_be_CO2=f1_4_be_CO2.fillna(value={"eprtr_sectorcode":'unknown','eprtr_sectorname':'unknown'})

In [34]:
f1_4_be_CO2.isnull().sum()

publicationdate                   0
countryname                       0
reportingyear                     0
eprtr_sectorcode                  0
eprtr_sectorname                  0
eprtranneximainactivity           1
facilityinspireid                 0
facilityname                      0
city                              7
longitude                         0
latitude                          0
addressconfidentialityreason    597
targetrelease                     0
pollutant                         0
emitted_co2_kg                    0
confidentialityreason           597
dtype: int64

In [35]:
f1_4_be_CO2.columns

Index(['publicationdate', 'countryname', 'reportingyear', 'eprtr_sectorcode',
       'eprtr_sectorname', 'eprtranneximainactivity', 'facilityinspireid',
       'facilityname', 'city', 'longitude', 'latitude',
       'addressconfidentialityreason', 'targetrelease', 'pollutant',
       'emitted_co2_kg', 'confidentialityreason'],
      dtype='str')

In [41]:
from src.silver import transform_data_f1_4

f1_4be,metric=transform_data_f1_4(f1_4_be_CO2,col_to_keep)


In [9]:
f1_4be.head()
print(f1_4be['facilityname'].str.len().max())
print(f1_4be['eprtr_sectorname'].str.len().max())

82
63


In [42]:
metric

{'table': 'silver_f1_4',
 'nb_null_in': np.int64(9),
 'nb_null_out': np.int64(1),
 'nb_col_out': 9}

In [43]:
df_monitoring=pd.DataFrame([metric])

In [44]:
df_monitoring.head()

,table,nb_null_in,nb_null_out,nb_col_out
0,silver_f1_4,9,1,9


In [45]:
f1_4be.isnull().sum()

reportingyear        0
eprtr_sectorcode     1
eprtr_sectorname     0
facilityinspireid    0
facilityname         0
city                 0
longitude            0
latitude             0
emitted_co2_kg       0
dtype: int64